In the gold layer, we joined our silver layer data to an external dataset to get the city names and exact pick-up and drop-off locations to be able to create the dashboard maps.

In [0]:
USE CATALOG taxi_data_gold;


In [0]:
CREATE SCHEMA IF NOT EXISTS gold_data;

In [0]:
CREATE OR REPLACE TABLE gold_data.ride_details
USING DELTA
AS

WITH pickup_details AS
(SELECT 
  t1.pickup_date, 
  t1.trip_dur_mins,
  t1.trip_distance,
  t1.fare_amount,
  cast(t1.pickup_zip AS string) AS pickup_zip,
  cast(t1.dropoff_zip AS string) AS dropoff_zip,
  t1.time_of_day AS pickup_time_of_day,
  t1.pickup_day_of_week,
  cast(t1.pickup_time AS string) AS pickup_time,
  cast(t1.dropoff_time AS string) AS dropoff_time,
  zip.lat AS pickup_lat,
  zip.lng AS pickup_lng,
  zip.city AS pickup_city, 
  zip.state_name AS pickup_state, zip.population AS piup_city_popltion,
  zip.density AS piup_city_pop_density,
  zip.county_name AS piup_city_county
FROM taxi_data_silver.bronze_transformed_data.taxi_data_transformed AS t1

-- Join to the external data source
LEFT JOIN taxi_data_bronze.data_ingestion.uszips AS zip                 
ON zip.zip = cast(t1.pickup_zip AS string)
ORDER BY pickup_date DESC)

SELECT pup.*,
  zip.lat AS drop_lat,
  zip.lng AS drop_lng,
  zip.city AS drop_city, 
  zip.state_name AS drop_state, zip.population AS drop_city_popltion,
  zip.density AS drop_city_pop_density,
  zip.county_name AS drop_city_county
FROM pickup_details AS pup
LEFT JOIN taxi_data_bronze.data_ingestion.uszips AS zip
ON pup.dropoff_zip = zip.zip;



Add the pickup latitude and longitudes to be able to locate them on map

In [0]:
CREATE OR REPLACE TEMPORARY VIEW pick_up_lat_long AS
SELECT
    gt.pickup_zip,
    zip.lat AS pickup_lat,
    zip.lng AS pickup_lng
FROM
    taxi_data_gold.gold_data.ride_details AS gt
LEFT JOIN
    taxi_data_bronze.data_ingestion.uszips AS zip
    ON gt.pickup_zip = zip.zip;

In [0]:
ALTER TABLE taxi_data_gold.gold_data.ride_details 
ADD COLUMN (
  pickup_lat DOUBLE,
  pickup_lng DOUBLE
);

In [0]:

MERGE INTO taxi_data_gold.gold_data.ride_details AS gt
USING (
  SELECT pickup_zip, pickup_lat, pickup_lng
  FROM (
    SELECT 
      pickup_zip, pickup_lat, pickup_lng,
      ROW_NUMBER() OVER (PARTITION BY pickup_zip ORDER BY pickup_zip) AS rn
    FROM pick_up_lat_long
  )
  WHERE rn = 1
) AS p
ON gt.pickup_zip = p.pickup_zip
WHEN MATCHED THEN
UPDATE SET 
  gt.pickup_lat = p.pickup_lat,
  gt.pickup_lng = p.pickup_lng;

Add the drop-off latitudes and longitudes of pickups to add map locations

In [0]:
CREATE OR REPLACE TEMPORARY VIEW drop_off_lat_long AS
SELECT
    gt.dropoff_zip,
    zip.lat AS drop_lat,
    zip.lng AS drop_lng
FROM
    taxi_data_gold.gold_data.ride_details AS gt
LEFT JOIN
    taxi_data_bronze.data_ingestion.uszips AS zip
    ON gt.dropoff_zip = zip.zip;

In [0]:
ALTER TABLE taxi_data_gold.gold_data.ride_details 
ADD COLUMN (
  drop_lat DOUBLE,
  drop_lng DOUBLE
);

In [0]:
MERGE INTO taxi_data_gold.gold_data.ride_details AS gt
USING (
  SELECT dropoff_zip, drop_lat, drop_lng
  FROM (
    SELECT 
      d.dropoff_zip, d.drop_lat, d.drop_lng,
      ROW_NUMBER() OVER (PARTITION BY d.dropoff_zip ORDER BY d.dropoff_zip) AS rn
    FROM drop_off_lat_long AS d
  )
  WHERE rn = 1
) AS p
ON gt.dropoff_zip = p.dropoff_zip

WHEN MATCHED THEN
UPDATE SET 
  gt.drop_lat = p.drop_lat,
  gt.drop_lng = p.drop_lng;
